# Constrained optimization

_______

## 1) Projected gradient

Projected gradient is easy when the optimization variable lies in a Cartesian-product space, for instance $x \in [0,1]^n$. the algorithm is simply:

$$ x_{k+1} = \mathcal P_{[0,1]^n}(x_k - \alpha d_x f(x) ) $$

In [ ]:
import numpy as np

def f(x, x0 = 1, y0 = 0.25):
    return (x[0]-x0)**2 + (x[1]-y0)**2

def cplxStep(x, f, h = 1e-50):
    dfdx = np.zeros_like(x)
    for i in range(len(dfdx)):
        e_i = np.zeros_like(x)
        e_i[i] = 1
        dfdx[i] = f(x + 1j * e_i * h).imag / h
    return dfdx

cplxStep([1,2], f, h = 1e-50)

In [ ]:
import matplotlib.pyplot as plt

def plot_background_square(f, lb = -0.5, ub = 0.5):
    X = np.linspace(lb*2, ub*2, 50)
    Y = np.linspace(lb*2, ub*2, 50)
    X, Y = np.meshgrid(X, Y)
    Z = f([X,Y])
    _, ax = plt.subplots()
    CS = ax.contour(X, Y, Z, np.arange(0,5,0.25))
    ax.clabel(CS, fontsize=10)
    plt.plot([lb,ub,ub,lb,lb],[lb,lb,ub,ub,lb], "r")
    ax.set_aspect('equal', 'box')

In [ ]:
def Px(x, lb=-0.5, ub=0.5):
    return np.minimum(ub, np.maximum(lb, x))

x0 = np.random.rand(2,10)
xProj = Px(x0)
plot_background_square(f, lb = -0.5, ub = 0.5)
plt.plot([x0[0], xProj[0]],[x0[1], xProj[1]], 'o--')
plt.show()

In [ ]:
def Pdfdx(x, d, proj = Px, h=1e-7):
    return (proj(x - d*h) - x)/h

def projected_descent(f, df_dx, x0, alpha = 1, maxit = 20, s = 0.1, tol = 1e-6):
    xList = [x0]
    dfdxList = []
    for _ in range(maxit):
        fx = f(xList[-1])
        dfdx = df_dx(xList[-1])
        dfdxList.append(dfdx)

        if np.linalg.norm(Pdfdx(xList[-1], dfdx)) < tol:
            break
        
        # update and projection
        xTest = Px(xList[-1] - alpha * dfdx)

        # Linesearch
        i =0
        while not(f(xTest) < fx + s * np.dot(dfdx, xTest - xList[-1])) and i < 100 :
            alpha /= 2
            i += 1
            xTest = Px(xList[-1] - alpha * dfdx)
        xList.append(xTest)

        # Basic step adaptation (default = increase, otherwise use linesearch)
        alpha *= 1.2

    return xList, dfdxList

def plot_path(xList):
    X = [x[0] for x in xList]
    Y = [x[1] for x in xList]
    plt.plot(X,Y,'--.')

In [ ]:
x0 = 2*np.random.rand(2)-1
df_dx = lambda x : cplxStep(x, f)
xList, dfdxList = projected_descent(f, df_dx, x0)
plot_background_square(f)
plot_path(xList)
plt.show()

In [ ]:
normDfDx = [np.linalg.norm(vec) for vec in dfdxList]
normDfDx_proj = [Pdfdx(x, d) for x, d in zip(xList, dfdxList)]

def plot_criterion(crit, label = ""):
    norm_crit = [np.linalg.norm(vec) for vec in crit]
    plt.semilogy(norm_crit, 'o--', label = label)
    plt.xlabel('iteration')
    plt.ylabel('Stop criterion')

plot_criterion(dfdxList, label = "norm of gradient")
plot_criterion(normDfDx_proj, label = "norm of projected gradient")
plt.grid(); plt.legend(); plt.show()

_____
## 2) Penalization

The principle is simple : adding a term to the objective function discouraging the optimizer to go there.

- **Interior penalization** : force the points to stay inside the admissible domain
- **Exterior penalization** : tolerate the points to go a little bit outside the admissible domain

In [ ]:
def penalization_interior(g, mu = 1):
    return - mu / g

def penalization_exterior(g, mu = 1):
    return mu * np.maximum(0, g) ** 2

g = np.linspace(-1,1,100)
plt.plot(g, penalization_interior(g))
plt.plot(g, penalization_exterior(g, 10))
plt.axis([-1,1,-1,10])

In [ ]:
def plot_background_g(f, g, R=0.5):
    X = np.linspace(-R*2, R*2, 50)
    Y = np.linspace(-R*2, R*2, 50)
    X, Y = np.meshgrid(X, Y)
    Z = f([X,Y])
    _, ax = plt.subplots()
    CS = ax.contour(X, Y, Z, np.arange(0,5,0.25))
    ax.clabel(CS, fontsize=10)
    ax.contour(X, Y, g([X,Y]), [0], colors = 'r')
    ax.set_aspect('equal', 'box')

In [ ]:
def g(x, R=0.5):
    return x[0]**2 + x[1]**2 - R**2

plot_background_g(f, g)

In [ ]:
def gradient_descent(f, df_dx, x0, alpha = 1, maxit = 20, s = 0.1, tol = 1e-6):
    xList = [x0]
    dfdxList = []
    for _ in range(maxit):
        fx = f(xList[-1])
        dfdx = df_dx(xList[-1])
        dfdxList.append(dfdx)
        if np.linalg.norm(dfdx) < tol:
            break
                
        # update
        xTest = xList[-1] - alpha * dfdx

        # Linesearch
        i =0
        while not(f(xTest) < fx + s * np.dot(dfdx, - alpha *dfdx)) and i < 100 :
            alpha /= 2
            i += 1
            xTest = xList[-1] - alpha * dfdx
        xList.append(xTest)

        # Basic step adaptation (default = increase, otherwise use linesearch)
        alpha *= 1.2

    return xList, dfdxList


In [ ]:
x0 = 2*np.random.rand(2)-1
fpext = lambda x: f(x) + penalization_exterior(g(x))
dfpext_dx = lambda x : cplxStep(x, fpext)
xList, dfdxList = gradient_descent(fpext, dfpext_dx, x0)
plot_background_g(fpext, g)
plot_path(xList)
plt.show()

In [ ]:
plot_criterion(dfdxList, label = "norm of gradient")
plt.grid()
plt.show()

In [ ]:
x0 = 0.5*np.random.rand(2)-0.25
mu = 0.1
fpint= lambda x: f(x) + mu*penalization_interior(g(x))
dfpint_dx = lambda x : cplxStep(x, fpint)
xList, dfdxList = gradient_descent(fpint, dfpint_dx, x0, alpha = 0.1)
plot_background_g(fpint, g)
plot_path(xList)
plt.show()

____
## 3) Augmented Lagrangian

Now we have :
$$ L(x,\lambda) = f(x) + \lambda g(x) + p(g(x))$$

With $p$ an exterior penalization function.
For an inequality constraint, $\lambda >0$. We want to find a saddle point of $L$, minimum in $x$ and maximum in $\lambda$. So we perform a gradient descent on $x$, then a hill climbing on $\lambda$.

In [ ]:
def augmented_lagrangian(f, g, x0, lam=0, penal = penalization_exterior, alpha = 1, maxit = 20, s = 0.1, tol = 1e-6):
    xList = [x0]
    for _ in range(maxit):
        L = lambda x : f(x) + lam * g(x) + penal(g(x))
        dLdx = lambda x : cplxStep(x, L)
        xList += gradient_descent(L, dLdx, xList[-1], alpha = alpha, maxit=maxit, s=s, tol=tol)[0]
        lam = Px(lam +  g(xList[-1]), lb = 0, ub = np.inf)
    return xList

In [ ]:
x0 = 2*np.random.rand(2)-1
xList = augmented_lagrangian(f, g, x0)
plot_background_g(f, g)
plot_path(xList)
plt.show()